<a href="https://colab.research.google.com/github/SonLee369/auto-labeling-scripts/blob/main/Segment_Auto_Labeling_Script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. CÀI ĐẶT THƯ VIỆN VÀ IMPORT
!pip install -q transformers torch torchvision Pillow numpy opencv-python

from transformers import Mask2FormerImageProcessor, Mask2FormerForUniversalSegmentation
from PIL import Image
import torch
import numpy as np
import cv2
import os
import shutil
import zipfile
from google.colab import files

# Bật GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng phần cứng: {device} (Khuyên dùng T4 GPU trên Colab)")

# ---------------------------------------------------------
# 2. TẢI MÔ HÌNH MASK2FORMER SEMANTIC
# ---------------------------------------------------------
print("Đang tải siêu mô hình Mask2Former Semantic Segmentation...")
ten_mo_hinh = "facebook/mask2former-swin-large-cityscapes-semantic"
processor = Mask2FormerImageProcessor.from_pretrained(ten_mo_hinh)
model = Mask2FormerForUniversalSegmentation.from_pretrained(ten_mo_hinh).to(device)

# ---------------------------------------------------------
# 3. DANH SÁCH 31 LABEL & MÃ MÀU HEX CHUẨN (KHÔNG BACKGROUND)
# ---------------------------------------------------------
raw_labels = [
    {"name": "pedestrian", "color": "#80e060"},
    {"name": "rider", "color": "#FF0000"},
    {"name": "car", "color": "#00008E"},
    {"name": "truck", "color": "#000046"},
    {"name": "bus", "color": "#003C64"},
    {"name": "train", "color": "#005064"},
    {"name": "motorcycle", "color": "#0000E6"},
    {"name": "bicycle", "color": "#770B20"},
    {"name": "traffic light", "color": "#d0a000"},
    {"name": "traffic sign", "color": "#502080"},
    {"name": "area/alternative", "color": "#62c4b2"},
    {"name": "area/drivable", "color": "#4a3d3c"},
    {"name": "lane/crosswalk", "color": "#7df28c"},
    {"name": "lane/double white", "color": "#d8c413"},
    {"name": "lane/double yellow", "color": "#ba3dfd"},
    {"name": "lane/road curb", "color": "#6e0189"},
    {"name": "lane/single other", "color": "#ff1f60"},
    {"name": "lane/single white", "color": "#a4faa4"},
    {"name": "lane/single yellow", "color": "#544df0"},
    {"name": "road", "color": "#804080"},
    {"name": "sidewalk", "color": "#F423E8"},
    {"name": "building", "color": "#464646"},
    {"name": "wall", "color": "#66669C"},
    {"name": "fence", "color": "#BE9999"},
    {"name": "pole", "color": "#999999"},
    {"name": "vegetation", "color": "#6B8E23"},
    {"name": "terrain", "color": "#98FB98"},
    {"name": "sky", "color": "#4682B4"},
    {"name": "person", "color": "#DC143C"},
    {"name": "traffic_light", "color": "#FAAA1E"},
    {"name": "traffic_sign", "color": "#DCDC00"}
]

def hex_to_rgb(hex_str):
    hex_str = hex_str.lstrip('#')
    return [int(hex_str[i:i+2], 16) for i in (0, 2, 4)]

# Chuyển mã Hex sang RGB
label_dict = {item["name"]: hex_to_rgb(item["color"]) for item in raw_labels}

# Mapping ID dự đoán của mô hình Cityscapes sang RGB
cityscapes_mapping = {
    0: label_dict["road"],
    1: label_dict["sidewalk"],
    2: label_dict["building"],
    3: label_dict["wall"],
    4: label_dict["fence"],
    5: label_dict["pole"],
    6: label_dict["traffic_light"],
    7: label_dict["traffic_sign"],
    8: label_dict["vegetation"],
    9: label_dict["terrain"],
    10: label_dict["sky"],
    11: label_dict["person"],
    12: label_dict["rider"],
    13: label_dict["car"],
    14: label_dict["truck"],
    15: label_dict["bus"],
    16: label_dict["train"],
    17: label_dict["motorcycle"],
    18: label_dict["bicycle"]
}

# Tạo ma trận bảng màu 256 ID
bang_pha_mau = np.zeros((256, 3), dtype=np.uint8)
for city_id, rgb in cityscapes_mapping.items():
    bang_pha_mau[city_id] = rgb

# ---------------------------------------------------------
# 4. CHUẨN BỊ THƯ MỤC VÀ FILE LABELMAP (KHÔNG BACKGROUND)
# ---------------------------------------------------------
thu_muc_luu = "SegmentationClass"
thu_muc_imagesets = "ImageSets/Segmentation"

if os.path.exists(thu_muc_luu): shutil.rmtree(thu_muc_luu)
if os.path.exists("ImageSets"): shutil.rmtree("ImageSets")
for f in os.listdir('.'):
    if f.lower().endswith(('.png', '.jpg', '.jpeg', '.zip', '.txt')) and f != "labelmap.txt":
        os.remove(f)

os.makedirs(thu_muc_luu)
os.makedirs(thu_muc_imagesets)

# Viết labelmap.txt CHỈ chứa 31 nhãn của bạn
with open("labelmap.txt", "w") as f:
    f.write("# label:color_rgb:parts:actions\n")
    for item in raw_labels:
        rgb = hex_to_rgb(item["color"])
        f.write(f"{item['name']}:{rgb[0]},{rgb[1]},{rgb[2]}::\n")

print("\nVui lòng chọn các ảnh gốc của bạn:")
uploaded = files.upload()

# ---------------------------------------------------------
# 5. XỬ LÝ VÀ TRÍCH XUẤT MASK
# ---------------------------------------------------------
print("\nBắt đầu nhờ AI vẽ nhãn (Mask2Former Semantic)...")
danh_sach_ten_anh = []

for filename in uploaded.keys():
    print(f"Đang phân tích: {filename}")

    image = Image.open(filename).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    target_sizes = [image.size[::-1]] # [height, width]
    predicted_mask = processor.post_process_semantic_segmentation(outputs, target_sizes=target_sizes)[0]
    ai_result = predicted_mask.cpu().numpy()

    # Nhúng bảng màu
    mask_anh_mau = bang_pha_mau[ai_result]
    mask_anh_mau_bgr = cv2.cvtColor(mask_anh_mau, cv2.COLOR_RGB2BGR)

    ten_file_goc, _ = os.path.splitext(filename)
    danh_sach_ten_anh.append(ten_file_goc)

    out_filename = f"{ten_file_goc}.png"
    cv2.imwrite(os.path.join(thu_muc_luu, out_filename), mask_anh_mau_bgr)
    print(f"  -> Đã tạo mask Semantic: {out_filename}")

with open(os.path.join(thu_muc_imagesets, "default.txt"), "w") as f:
    for ten in danh_sach_ten_anh: f.write(f"{ten}\n")

# ---------------------------------------------------------
# 6. ĐÓNG GÓI VÀ TẢI VỀ
# ---------------------------------------------------------
print("\nĐang đóng gói thành file ZIP chuẩn CVAT 1.1...")
zip_filename = "CVAT_Semantic_31Classes.zip"

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    zipf.write("labelmap.txt")

    for root, dirs, files_list in os.walk(thu_muc_luu):
        for file in files_list:
            zipf.write(os.path.join(root, file))

    for root, dirs, files_list in os.walk("ImageSets"):
        for file in files_list:
            zipf.write(os.path.join(root, file))

print("Xong! Đang tải file ZIP về máy tính của bạn...")
files.download(zip_filename)